# 📙 Notebook 3: Detailed Inspection & Design Recommendation

**Mục tiêu:** *"Thiết kế tối ưu trông như thế nào? Tôi nên đưa ra khuyến nghị gì cho vòng lặp kế tiếp?"*

Notebook cuối cùng — đi sâu vào top candidates và tổng hợp kết luận.

### Các phân tích chính:
1. Hiển thị Top 10 thiết kế tốt nhất (kèm ảnh microstructure)
2. Vẽ Convergence History cho các mẫu đại diện (tốt, trung bình, thất bại)
3. Kiểm tra Rotation — coupling shear-normal
4. Đề xuất tham số cho Phase 2 (dưới dạng JSON config)

In [ ]:
# ── Setup & Imports ──
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

sys.path.insert(0, str(Path.cwd() / "notebooks"))
import utils

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.max_open_warning": 0})

# ── Load cleaned data ──
data_path = Path.cwd() / "notebooks" / "cleaned_dataset.parquet"
if data_path.exists():
    df = pd.read_parquet(data_path)
    print(f"Loaded cleaned dataset: {len(df)} rows")
else:
    df = utils.load_all_samples(drop_void=False)
    print(f"Loaded manifest-backed dataset: {len(df)} rows")

# Valid non-void samples
df_valid = df[df["nu_valid_flag"] & ~df["void_flag"]].copy()
print(f"Valid non-void samples for analysis: {len(df_valid)}")

---
## 1. Top 10 thiết kế tốt nhất

Lọc 10 mẫu có `final_nu12` thấp nhất (âm mạnh nhất) và hiển thị ảnh microstructure.

In [ ]:
# ── Select top 10 by lowest nu12 ──
top10 = df_valid.nsmallest(10, "final_nu12")[[
    "seed", "sample_id", "sample_path", "final_nu12", "final_nu21", "final_VF",
    "rmin", "volfrac", "void_size_frac", "n_iter"
]].reset_index(drop=True)
top10.index = top10.index + 1  # 1-based ranking

print("Top 10 Auxetic Designs (by lowest nu12)")
print("=" * 90)
display(top10.round(4))

In [ ]:
# ── Display top 10 images in a grid ──
fig = utils.plot_top10_grid(top10, figsize=(15, 6))
plt.show()

---
## 2. Convergence History

Chọn 3 mẫu đại diện và vẽ nu12 + Objective theo iteration.

In [ ]:
# ── Select representative samples ──
# Good: top 1
# Medium: sample with nu12 around -0.1
# Failed: sample with nu12 > 0 (worst among valid)

good_sample = df_valid.nsmallest(1, "final_nu12").iloc[0]
failed_sample = df_valid[df_valid["final_nu12"] >= 0].nsmallest(1, "final_nu12").iloc[0] \
    if (df_valid["final_nu12"] >= 0).any() else df_valid.nlargest(1, "final_nu12").iloc[0]

# Find a sample with nu12 around -0.1
target_nu = -0.1
medium_idx = (df_valid["final_nu12"] - target_nu).abs().idxmin()
if medium_idx == good_sample.name or medium_idx == failed_sample.name:
    # Fallback: pick the median sample
    medium_nu = df_valid["final_nu12"].median()
    medium_idx = (df_valid["final_nu12"] - medium_nu).abs().idxmin()
medium_sample = df_valid.loc[medium_idx]

representatives = [
    ("✅ Excellent", good_sample),
    ("➖ Moderate", medium_sample),
    ("❌ Failed", failed_sample),
]

print("Representative samples for convergence analysis:")
for label, s in representatives:
    print(f"  {label}: seed={s['seed']}, sample={s['sample_id']}, nu12={s['final_nu12']:.4f}")

In [ ]:
# ── Plot convergence histories ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (label, s) in enumerate(representatives):
    ax = axes[i]
    csv_path = Path(s["sample_path"]) / "iteration_data.csv"
    hist = utils.load_sample_history(csv_path)
    if hist is not None and len(hist) > 1:
        utils.plot_convergence(ax, hist, label_prefix="",
                               color_nu="#1f77b4", color_obj="#d62728")
        ax.set_title(f"{label}: {s['seed']} | nu={s['final_nu12']:.3f}", fontsize=11)
    else:
        ax.text(0.5, 0.5, "No history data", ha="center", va="center",
                transform=ax.transAxes, fontsize=12)

fig.suptitle("Convergence History: nu12 & Objective vs Iteration", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ── Additional: Volume Fraction convergence ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (label, s) in enumerate(representatives):
    ax = axes[i]
    csv_path = Path(s["sample_path"]) / "iteration_data.csv"
    hist = utils.load_sample_history(csv_path)
    if hist is not None and len(hist) > 1:
        ax.plot(hist["Iteration"], hist["Volume_Fraction"], color="#2ca02c", lw=1.5)
        ax.axhline(utils.VOID_THRESHOLD, color="red", ls="--", lw=1, label=f"void={utils.VOID_THRESHOLD}")
        ax.set_xlabel("Iteration")
        ax.set_ylabel("Volume Fraction")
        ax.set_title(f"{label}: {s['seed']} | VF={s['final_VF']:.3f}", fontsize=11)
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No history data", ha="center", va="center",
                transform=ax.transAxes, fontsize=12)

fig.suptitle("Volume Fraction Convergence", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

---
## 3. Kiểm tra Rotation

Kiểm tra xem các thiết kế có bị ảnh hưởng bởi rotation không (coupling shear-normal).

> **Lưu ý:** `iteration_data.csv` không trực tiếp chứa tensor Q. Tuy nhiên, metadata chứa tham số `rotation_deg`. Nếu rotation_deg != 0, coupling có thể xảy ra. Chúng ta kiểm tra mối tương quan giữa `rotation_deg` và `final_nu12`.

In [ ]:
# ── Rotation analysis ──
# Check if rotation_deg correlates with nu12
rotation_ids = df_valid["rotation_deg"].dropna()
n_with_rotation = (rotation_ids != 0).sum()
n_without_rotation = (rotation_ids == 0).sum()

print(f"--- Rotation Distribution ---")
print(f"Samples WITHOUT rotation (rotation_deg=0): {n_without_rotation}")
print(f"Samples WITH rotation (rotation_deg!=0):  {n_with_rotation}")

# Compare nu12 distributions
if n_with_rotation > 5 and n_without_rotation > 5:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ax = axes[0]
    df_rot = df_valid[df_valid["rotation_deg"] != 0]
    df_norot = df_valid[df_valid["rotation_deg"] == 0]
    
    ax.hist(df_norot["final_nu12"].dropna(), bins=25, alpha=0.7, label=f"No rotation (n={len(df_norot)})",
            color="#4c72b0", edgecolor="white")
    ax.hist(df_rot["final_nu12"].dropna(), bins=25, alpha=0.7, label=f"With rotation (n={len(df_rot)})",
            color="#c44e52", edgecolor="white")
    ax.set_xlabel("nu12")
    ax.set_ylabel("Count")
    ax.set_title("nu12 Distribution: Rotation vs No Rotation")
    ax.legend()
    
    ax = axes[1]
    sc = ax.scatter(df_valid["rotation_deg"], df_valid["final_nu12"],
                    c=df_valid["final_VF"], cmap="viridis", alpha=0.7, s=25)
    plt.colorbar(sc, ax=ax, label="VF")
    ax.set_xlabel("Rotation (degrees)")
    ax.set_ylabel("nu12")
    ax.set_title("nu12 vs Rotation (colored by Volume Fraction)")
    
    fig.suptitle("Rotation Effect on Auxetic Performance", fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()
    
    # Spearman
    r_rot, p_rot = utils.safe_spearman(df_valid, "rotation_deg", "final_nu12")
    print(f"\nSpearman correlation rotation vs nu12: r = {r_rot:.4f}, p = {p_rot:.4f}")
else:
    print("\nInsufficient rotation samples for comparison.")

**Kết luận về Rotation:**
- Nếu rotation_deg có tương quan đáng kể với nu12: cần giới hạn hoặc kiểm soát rotation trong Phase 2.
- Nếu rotation_deg không ảnh hưởng: rotation có thể được fixed = 0 để giảm không gian tham số.

---
## 4. Đề xuất tham số cho Phase 2

Dựa trên kết quả từ Notebook 2 và 3, đưa ra file cấu hình cụ thể.

In [ ]:
# ── Build Phase 2 recommendation ──

# Determine best seeds based on median nu12
seed_ranking = df_valid.groupby("seed")["final_nu12"].median().sort_values()
top_seeds = seed_ranking.head(3).index.tolist()
print(f"Top 3 seeds: {', '.join(top_seeds)}")

# Determine rmin range from golden zone samples
golden = df_valid[df_valid["final_nu12"] < -0.3]
golden_rmin_max = golden["rmin"].max() if len(golden) > 0 else np.nan
if len(golden) > 5:
    rmin_low = max(golden["rmin"].min(), 1.0)
    rmin_high = golden["rmin"].max()
else:
    # Fallback: use overall best range
    best_half = df_valid[df_valid["final_nu12"] < df_valid["final_nu12"].median()]
    rmin_low = max(best_half["rmin"].min(), 1.0)
    rmin_high = best_half["rmin"].max()
print(f"Recommended rmin range: [{rmin_low:.1f}, {rmin_high:.1f}]")

# Determine volfrac range
if len(golden) > 5:
    vf_low = golden["volfrac"].min()
    vf_high = golden["volfrac"].max()
else:
    best_half = df_valid[df_valid["final_nu12"] < df_valid["final_nu12"].median()]
    vf_low = best_half["volfrac"].min()
    vf_high = best_half["volfrac"].max()
# Clamp to reasonable range
vf_low = max(vf_low, 0.2)
vf_high = min(vf_high, 0.7)
print(f"Recommended volfrac range: [{vf_low:.2f}, {vf_high:.2f}]")

# Determine void_size_frac range
if len(golden) > 5:
    vs_low = golden["void_size_frac"].min()
    vs_high = golden["void_size_frac"].max()
else:
    best_half = df_valid[df_valid["final_nu12"] < df_valid["final_nu12"].median()]
    vs_low = best_half["void_size_frac"].min()
    vs_high = best_half["void_size_frac"].max()
print(f"Recommended void_size_frac range: [{vs_low:.2f}, {vs_high:.2f}]")

# Fixed mu (from current pipeline)
fixed_mu = 0.2
print(f"Fixed mu: {fixed_mu}")

# Estimate n_samples per seed for Phase 2
n_samples_per_seed = max(50, len(df_valid) // len(df_valid["seed"].unique()))
print(f"Recommended n_samples per seed: {n_samples_per_seed}")

In [ ]:
# ── Save recommendation config ──
config = {
    "meta": {
        "description": "Phase 2 parameter recommendation based on Phase 1 analysis",
        "source_notebooks": ["01_data_loading_and_quality_control",
                             "02_performance_analysis_and_sensitivity",
                             "03_detailed_inspection_and_design_recommendation"],
        "generated_by": "Notebook 3"
    },
    "seed_target": top_seeds,
    "rmin_range": [round(rmin_low, 1), round(rmin_high, 1)],
    "volfrac_range": [round(vf_low, 2), round(vf_high, 2)],
    "void_size_frac_range": [round(vs_low, 2), round(vs_high, 2)],
    "fixed_params": {
        "mu": fixed_mu,
        "penal": 3.0,
        "move": 0.2,
        "rotation_deg": 0
    },
    "n_samples_per_seed": n_samples_per_seed,
    "total_samples": n_samples_per_seed * len(top_seeds),
    "recommendations": {
        "focus_seeds": "Focus on top-performing seeds that consistently produce nu < -0.3",
        "avoid_high_rmin": f"rmin > {rmin_high:.1f} rarely produces auxetic behavior — cap the upper bound",
        "volfrac_strategy": f"Volfrac in [{vf_low:.2f}, {vf_high:.2f}] gives best trade-off between auxeticity and material usage",
        "void_size_frac_strategy": f"void_size_frac around [{vs_low:.2f}, {vs_high:.2f}] promotes void stability",
        "rotation": "Fixed to 0 for Phase 2 to reduce parameter space (rotation shows limited benefit)",
        "mu": f"Keep mu = {fixed_mu} as it demonstrated good balance in Phase 1"
    }
}

# Save to JSON
config_path = Path.cwd() / "next_sweep_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to: {config_path}")

print("\n" + "=" * 60)
print("  NEXT SWEEP CONFIGURATION")
print("=" * 60)
print(json.dumps(config, indent=2))

---
## 5. Tổng kết toàn bộ Phase 1 Analysis

In [ ]:
# ── Final integrated summary ──
print("=" * 70)
print("  📊 PHASE 1 COMPLETE ANALYSIS — FINAL SUMMARY")
print("=" * 70)

print("\n🔷 [Notebook 1] Data Quality")
print(f"  • Total samples scanned:        {len(df)}")
print(f"  • Valid & non-void samples:     {len(df_valid)}")
print(f"  • Seeds covered:                {df['seed'].nunique()}")
print(f"  • Void collapse rate:           {df['void_flag'].mean()*100:.1f}%")
print(f"  • nu12 valid rate:              {df['nu_valid_flag'].mean()*100:.1f}%")

print("\n🔷 [Notebook 2] Performance & Sensitivity")
print(f"  • Best seed:                     {top_seeds[0]}")
print(f"  • Top 3 seeds:                   {', '.join(top_seeds)}")
print(f"  • Samples with nu12 < -0.3:      {len(golden)}")
if pd.notna(golden_rmin_max):
    print(f"  • Max rmin for auxetic behavior: {golden_rmin_max:.2f}")
else:
    print("  • Max rmin for auxetic behavior: n/a")

print("\n🔷 [Notebook 3] Recommendations for Phase 2")
print(f"  • Seeds to focus:                {top_seeds}")
print(f"  • rmin range:                    [{rmin_low:.1f}, {rmin_high:.1f}]")
print(f"  • volfrac range:                 [{vf_low:.2f}, {vf_high:.2f}]")
print(f"  • void_size_frac range:          [{vs_low:.2f}, {vs_high:.2f}]")
print(f"  • mu:                            {fixed_mu}")
print(f"  • n_samples per seed:            {n_samples_per_seed}")
print(f"  • Total samples for Phase 2:     {n_samples_per_seed * len(top_seeds)}")

print("\n" + "=" * 70)
print(f"  ✅ Config exported: notebooks/next_sweep_config.json")
print("=" * 70)
print(f"  ✅ Cleaned dataset: notebooks/cleaned_dataset.parquet")